# 与信審査① — ベースラインをテストで測る

出典：第01章 03節／パス：`01_credit/01_baseline.ipynb`／パッケージ：numpy・pandas・scikit-learn・matplotlib／実行時間の目安：約1分

このノート単体で実行できる。German Credit（UCI公開データ）を取得して使う。取得できないときは `input/applications.csv`（German Credit形式の合成代替）、それも無いときは内蔵の生成関数へ切り替わる。合成代替では数値が本文と変わる。
機微な属性（A9 性別・婚姻状況、A13 年齢、A20 外国人労働者）は特徴量から外す。改善モデルのコードは `02_improved.ipynb` に分けてある。最終評価はテストセットで1回だけ行う。

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score

RS = 0


def make_credit_data(n: int, seed: int = 42) -> pd.DataFrame:
    """German Credit と同じ列（A1〜A20）の合成代替。値の分布は実データの近似、貸倒との関係は仮置き。"""
    r = np.random.default_rng(seed)

    def cat(codes, p):
        p = np.asarray(p, dtype=float)
        return r.choice(codes, size=n, p=p / p.sum())

    d = pd.DataFrame({
        "A1": cat(["A11", "A12", "A13", "A14"], [27, 27, 6, 40]),
        "A2": np.clip(np.round(r.gamma(3.0, 7.0, size=n)), 4, 72).astype(int),
        "A3": cat(["A30", "A31", "A32", "A33", "A34"], [4, 5, 53, 9, 29]),
        "A4": cat(["A40", "A41", "A42", "A43", "A44", "A45", "A46", "A48", "A49", "A410"],
                  [23, 10, 18, 28, 1, 2, 5, 1, 10, 1]),
        "A5": np.clip(np.round(np.exp(r.normal(7.8, 0.75, size=n))), 250, 18424).astype(int),
        "A6": cat(["A61", "A62", "A63", "A64", "A65"], [60, 10, 6, 5, 18]),
        "A7": cat(["A71", "A72", "A73", "A74", "A75"], [6, 17, 34, 17, 25]),
        "A8": cat([1, 2, 3, 4], [14, 23, 16, 47]),
        "A9": cat(["A91", "A92", "A93", "A94"], [5, 31, 55, 9]),
        "A10": cat(["A101", "A102", "A103"], [91, 4, 5]),
        "A11": cat([1, 2, 3, 4], [13, 31, 15, 41]),
        "A12": cat(["A121", "A122", "A123", "A124"], [28, 23, 33, 15]),
        "A13": np.clip(np.round(r.gamma(2.5, 6.6, size=n) + 19), 19, 75).astype(int),
        "A14": cat(["A141", "A142", "A143"], [14, 5, 81]),
        "A15": cat(["A151", "A152", "A153"], [18, 71, 11]),
        "A16": cat([1, 2, 3, 4], [63, 33, 3, 1]),
        "A17": cat(["A171", "A172", "A173", "A174"], [2, 20, 63, 15]),
        "A18": cat([1, 2], [85, 15]),
        "A19": cat(["A191", "A192"], [60, 40]),
        "A20": cat(["A201", "A202"], [96, 4]),
    })
    # 貸倒の起こりやすさ（仮置き）：口座状態・期間・返済履歴・金額・貯蓄・資産・他ローンで決まる
    logit = (
        -1.05
        + d["A1"].map({"A11": 1.0, "A12": 0.5, "A13": -0.2, "A14": -1.2})
        + 0.03 * (d["A2"] - 21)
        + d["A3"].map({"A30": 1.0, "A31": 0.9, "A32": 0.0, "A33": -0.2, "A34": -0.7})
        + 0.00008 * (d["A5"] - 3271)
        + d["A6"].map({"A61": 0.3, "A62": 0.0, "A63": -0.3, "A64": -0.5, "A65": -0.4})
        + 0.4 * (d["A12"] == "A124") + 0.4 * (d["A14"] == "A141")
        + 0.15 * (d["A8"] - 3)
    )
    d["default"] = (r.uniform(size=n) < 1.0 / (1.0 + np.exp(-logit))).astype(int)
    return d


GERMAN_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
GERMAN_COLS = ["A1", "A2", "A3", "A4", "A5", "A6", "A7", "A8", "A9", "A10",
               "A11", "A12", "A13", "A14", "A15", "A16", "A17", "A18", "A19", "A20", "klass"]
GERMAN_NUM = ["A2", "A5", "A8", "A11", "A13", "A16", "A18"]
# 機微な属性（事前指定で特徴量から外す）：A9=性別・婚姻状況、A13=年齢、A20=外国人労働者
SENSITIVE = ["A9", "A13", "A20"]


def load_german() -> pd.DataFrame:
    g = pd.read_csv(GERMAN_URL, sep=r"\s+", header=None, names=GERMAN_COLS)
    g["default"] = (g["klass"] == 2).astype(int)  # 2=信用リスクが悪い（bad）を1とする
    return g.drop(columns=["klass"])


INPUT = "input/applications.csv"
try:
    df = load_german()
    print("公開データを使用: German Credit（個人与信の実データ）")
except Exception:
    try:
        df = pd.read_csv(INPUT)
        print(f"Inputを使用: {INPUT}（German Credit形式の合成代替）")
    except FileNotFoundError:
        df = make_credit_data(1000)
        print("Inputが無いため合成データを生成（German Credit形式）")
FEATS = [c for c in df.columns if c not in SENSITIVE + ["default"]]
num_cols = [c for c in FEATS if c in GERMAN_NUM]
cat_cols = [c for c in FEATS if c not in GERMAN_NUM]
print(f"件数: {len(df)}, 貸倒率: {df['default'].mean():.3f} / 特徴量 {len(FEATS)}列（機微な属性{len(SENSITIVE)}列を除外）")

公開データを使用: German Credit（個人与信の実データ）
件数: 1000, 貸倒率: 0.300 / 特徴量 17列（機微な属性3列を除外）


In [2]:
X = df.drop(columns=["default"])
y = df["default"]
# 最終評価用のテストを先に分ける（調整には使わない）。検証は学習データの中の層化5分割CVで行う。
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RS
)
pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
print(f"学習: {len(X_tr)}件（検証は学習内の5分割CV） / テスト: {len(X_te)}件")
print(f"貸倒率 学習:{y_tr.mean():.3f} テスト:{y_te.mean():.3f}")

学習: 750件（検証は学習内の5分割CV） / テスト: 250件
貸倒率 学習:0.300 テスト:0.300


In [3]:
# ベースライン①：陽性率で振る物差し（最終評価はテストで1回だけ）
dummy = DummyClassifier(strategy="prior").fit(X_tr, y_tr)
p_dummy = dummy.predict_proba(X_te)[:, 1]
print(f"[ダミー] PR-AUC={average_precision_score(y_te, p_dummy):.4f} / ROC-AUC={roc_auc_score(y_te, p_dummy):.4f}")

# ベースライン②：説明しやすい線形の物差し
lr = make_pipeline(pre, LogisticRegression(max_iter=5000)).fit(X_tr, y_tr)
p_lr = lr.predict_proba(X_te)[:, 1]
print(f"[ロジスティック回帰] PR-AUC={average_precision_score(y_te, p_lr):.4f} / ROC-AUC={roc_auc_score(y_te, p_lr):.4f}")

[ダミー] PR-AUC=0.3000 / ROC-AUC=0.5000
[ロジスティック回帰] PR-AUC=0.5999 / ROC-AUC=0.7840


In [4]:
# 閾値0.5で切ったときの素顔（この閾値が最適という意味ではない。閾値は検証で決める）
COST_FN, COST_FP = 10.0, 2.0  # 仮置き（万円）：見逃し（貸倒）10万円、空振り（機会損失）2万円＝5:1
cm = confusion_matrix(y_te, (p_lr >= 0.5).astype(int))
print("混同行列（行=正解 0/1, 列=予測 0/1, 閾値0.5・ベースライン）:")
print(cm)
tn, fp, fn, tp = cm.ravel()
print(f"適合率: {tp / (tp + fp):.3f} / 再現率: {tp / (tp + fn):.3f}")
print(f"期待損失（万円/件）: 閾値0.5 {(fn * COST_FN + fp * COST_FP) / len(y_te):.3f}"
      f" / 全件承認 {y_te.sum() * COST_FN / len(y_te):.3f} / 全件否決 {(1 - y_te).sum() * COST_FP / len(y_te):.3f}")
assert np.isfinite(p_lr).all()

混同行列（行=正解 0/1, 列=予測 0/1, 閾値0.5・ベースライン）:
[[159  16]
 [ 46  29]]
適合率: 0.644 / 再現率: 0.387
期待損失（万円/件）: 閾値0.5 1.968 / 全件承認 3.000 / 全件否決 1.400


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
_book_fig = Path("../../../books/python-machine-jitumu/assets/images")
FIG = str(_book_fig if _book_fig.is_dir() else Path("outputs"))
Path(FIG).mkdir(parents=True, exist_ok=True)
from sklearn.metrics import precision_recall_curve
plt.figure()
for name, p in [("dummy", p_dummy), ("logreg", p_lr)]:
    prec, rec, _ = precision_recall_curve(y_te, p)
    plt.step(rec, prec, where="post", label=name)
plt.xlabel("recall"); plt.ylabel("precision"); plt.legend(); plt.title("PR curves (test)")
plt.savefig(f"{FIG}/code-result-j01-01.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved code-result-j01-01.png")

Matplotlib is building the font cache; this may take a moment.


saved code-result-j01-01.png


## 読み取りどころ

- ダミーのPR-AUCが貸倒率とほぼ一致すること。
- 線形のPR-AUCがベースラインの物差しになること（改善幅は `02_improved.ipynb` で測る）。
- 閾値0.5の期待損失を、モデルを使わない方針（全件承認・全件否決）と比べること。
- テストは最終評価用に残し、閾値の調整には使わないこと。